# Prompt-only GPT-4o Schema Conformance Experiment

This notebook compares a plain prompt-only GPT-4o extraction baseline against the existing BAML extraction outputs. The purpose is to test whether prompt instruction alone can produce JSON outputs that are structurally valid, ontology-conformant, and ready for Neo4j ingestion.

## Step 1: Setup

Run this cell first. It loads the project paths, OpenAI client, and helper functions used for prompt-only extraction and schema validation.

In [ ]:
import base64
import json
import os
import re
import time
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(*args, **kwargs):
        return False

from openai import OpenAI

load_dotenv()

repo_root = Path.cwd()

if repo_root.name == "knowledge_extraction":
    repo_root = repo_root.parent.parent

DATA_DIR = repo_root / "data" / "processed"
IMAGE_DIR = repo_root / "data" / "raw" / "Manual Book" / "Images"
OUTPUT_DIR = repo_root / "data" / "processed"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("Repo root:", repo_root)
print("Data dir:", DATA_DIR)
print("Image dir:", IMAGE_DIR)
print("Output dir:", OUTPUT_DIR)
print("OPENAI_API_KEY loaded:", os.getenv("OPENAI_API_KEY") is not None)

## Step 2: Check Input Data

This confirms the number of source maintenance-log cases and manual-book images that will be used in the prompt-only experiment.

In [ ]:
def count_jsonl(path):
    with open(path, "r", encoding="utf-8") as file:
        return sum(1 for line in file if line.strip())

input_20 = DATA_DIR / "20_cases_for_baml_fewshot.jsonl"
input_remaining = DATA_DIR / "remaining_cases_for_baml_fewshot.jsonl"
manual_images = sorted(IMAGE_DIR.glob("*.jpg")) + sorted(IMAGE_DIR.glob("*.jpeg")) + sorted(IMAGE_DIR.glob("*.png"))

print("20-case evaluation logs:", count_jsonl(input_20))
print("Remaining maintenance logs:", count_jsonl(input_remaining))
print("Manual-book images:", len(manual_images))
print("Manual-book image names:")
for image in manual_images:
    print("-", image.name)

Define Schema Validation Logic

In [ ]:
EXPECTED_FIELDS_PER_CASE = 6
ENUM_CHECKS_PER_CASE = 2

VALID_MACHINES = {"IBM3", "IBM4"}
VALID_RESOLUTION = {"Resolved", "Unknown", "N/A"}

EXPECTED_TOP_KEYS = {
    "fault_location",
    "fault_symptoms",
    "fault_reason",
    "fault_measures",
    "resolution_status",
}

EXPECTED_LOCATION_KEYS = {"name", "machine"}


def analyze_schema(result):
    missing = 0
    extra = 0
    type_error = 0
    enum_violation = 0

    if not isinstance(result, dict):
        return EXPECTED_FIELDS_PER_CASE, 0, EXPECTED_FIELDS_PER_CASE, ENUM_CHECKS_PER_CASE

    for key in EXPECTED_TOP_KEYS:
        if key not in result:
            missing += 1

    for key in result:
        if key not in EXPECTED_TOP_KEYS:
            extra += 1

    location = result.get("fault_location")

    if not isinstance(location, dict):
        missing += 2
        type_error += 2
        enum_violation += 1
    else:
        if "name" not in location:
            missing += 1
        elif not isinstance(location["name"], str):
            type_error += 1

        if "machine" not in location:
            missing += 1
        else:
            machine = location["machine"]

            if machine is not None and not isinstance(machine, str):
                type_error += 1
            elif machine is not None and machine not in VALID_MACHINES:
                enum_violation += 1

        for key in location:
            if key not in EXPECTED_LOCATION_KEYS:
                extra += 1

    symptoms = result.get("fault_symptoms")

    if symptoms is None:
        missing += 1
    elif not isinstance(symptoms, list):
        type_error += 1
    else:
        for symptom in symptoms:
            if not isinstance(symptom, str):
                type_error += 1

    reasons = result.get("fault_reason")

    if reasons is None:
        missing += 1
    elif not isinstance(reasons, list):
        type_error += 1
    else:
        for reason in reasons:
            if not isinstance(reason, dict):
                type_error += 1
            elif "name" not in reason or not isinstance(reason["name"], str):
                type_error += 1

            if isinstance(reason, dict):
                for key in reason:
                    if key != "name":
                        extra += 1

    measures = result.get("fault_measures")

    if measures is None:
        missing += 1
    elif not isinstance(measures, list):
        type_error += 1
    else:
        for measure in measures:
            if not isinstance(measure, dict):
                type_error += 1
            elif "description" not in measure or not isinstance(measure["description"], str):
                type_error += 1

            if isinstance(measure, dict):
                for key in measure:
                    if key != "description":
                        extra += 1

    resolution = result.get("resolution_status")

    if resolution is None:
        missing += 1
    elif not isinstance(resolution, str):
        type_error += 1
    elif resolution not in VALID_RESOLUTION:
        enum_violation += 1

    return missing, extra, type_error, enum_violation


def is_neo4j_ingestable(report):
    missing, extra, type_error, enum_violation = analyze_schema(report)

    if missing or extra or type_error or enum_violation:
        return False

    location = report.get("fault_location", {})
    name = location.get("name")

    return isinstance(name, str) and bool(name.strip())

Define Prompt-only GPT-4o Extraction Functions

In [ ]:
MODEL = "gpt-4o"


def load_jsonl(path):
    rows = []

    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

    return rows


def case_to_log_text(case):
    if isinstance(case, str):
        return case

    return case.get("text", "")


def case_id_from_log(log_text, fallback):
    first_line = log_text.splitlines()[0].strip() if log_text else ""

    if first_line.startswith("Case-ID:"):
        return first_line.replace("Case-ID:", "").strip()

    return fallback


def clean_json_text(text):
    if text is None:
        return ""

    text = text.strip()
    text = re.sub(r"^```(?:json)?", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"```$", "", text).strip()

    return text


def parse_json_or_none(text):
    try:
        return json.loads(clean_json_text(text))
    except Exception:
        return None


def build_log_prompt(log_text):
    return f"""
You are an expert in analyzing technical maintenance logs.

Extract the information from this maintenance log.

Return the answer in JSON format.

The JSON should contain:
- fault_location: an object containing the component name and the machine name if mentioned.
- fault_symptoms: a list of text strings describing observable problems.
- fault_reason: a list of objects, where each object contains the stated cause of the fault.
- fault_measures: a list of objects, where each object contains a maintenance action or corrective measure.
- resolution_status: a text value indicating whether the fault was resolved, unknown, or N/A.

---

{log_text}
---
""".strip()


def build_manual_prompt(source_image):
    return """
You are an expert in industrial machine maintenance and repair.

The image is a troubleshooting table from a machine manual.

Return exactly one JSON array. Each item must follow this schema:
- fault_location: an object containing the component name and the machine name if mentioned.
- fault_symptoms: a list of text strings describing observable problems.
- fault_reason: a list of objects, where each object contains the stated cause of the fault.
- fault_measures: a list of objects, where each object contains a maintenance action or corrective measure.
- resolution_status: a text value indicating whether the fault was resolved, unknown, or N/A.

""".strip()


def image_to_data_url(path):
    suffix = path.suffix.lower()
    mime = "image/png" if suffix == ".png" else "image/jpeg"

    encoded = base64.b64encode(path.read_bytes()).decode("utf-8")

    return f"data:{mime};base64,{encoded}"


def call_gpt4o(messages, max_retries=3):
    last_error = None

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                temperature=0,
                messages=messages,
            )

            return response.choices[0].message.content

        except Exception as exc:
            last_error = exc
            wait_time = 2 ** attempt
            print(f"Error: {exc}. Retrying in {wait_time} seconds...")
            time.sleep(wait_time)

    raise last_error

Run Prompt-only Extraction on 20 Evaluation Cases

In [ ]:
def extract_logs(input_filename, output_filename):
    cases = load_jsonl(DATA_DIR / input_filename)
    output_path = OUTPUT_DIR / output_filename

    if output_path.exists():
        results = json.loads(output_path.read_text(encoding="utf-8"))
        print(f"Resuming from existing file: {len(results)} already processed")
    else:
        results = []

    for idx, case in enumerate(cases[len(results):], start=len(results) + 1):
        log_text = case_to_log_text(case)
        case_id = case_id_from_log(log_text, f"case_{idx}")

        print(f"Processing {idx}/{len(cases)}: {case_id}")

        try:
            raw_output = call_gpt4o([
                {
                    "role": "system",
                    "content": "You extract structured JSON from maintenance logs."
                },
                {
                    "role": "user",
                    "content": build_log_prompt(log_text)
                }
            ])

            parsed_output = parse_json_or_none(raw_output)

            results.append({
                "case_id": case_id,
                "result": parsed_output
            })

        except Exception as exc:
            print("Error:", exc)

            results.append({
                "case_id": case_id,
                "result": None
            })

        output_path.write_text(
            json.dumps(results, indent=2, ensure_ascii=False),
            encoding="utf-8"
        )

    return output_path

In [ ]:
prompt_20_path = extract_logs(
    input_filename="20_cases_for_baml_fewshot.jsonl",
    output_filename="prompt_only_gpt4o_20_cases_plain.json",
)

print("Saved:", prompt_20_path)

schema evaluator

In [ ]:
import json
from pathlib import Path

repo_root = Path.cwd()

if repo_root.name == "knowledge_extraction":
    repo_root = repo_root.parent.parent

prompt_file = repo_root / "data" / "processed" / "prompt_only_gpt4o_20_cases_plain.json"

print("File exists:", prompt_file.exists())
print("File path:", prompt_file)

In [ ]:
EXPECTED_FIELDS_PER_CASE = 6
ENUM_CHECKS_PER_CASE = 2

VALID_MACHINES = {"IBM3", "IBM4"}
VALID_RESOLUTION = {"Resolved", "Unknown", "N/A"}

EXPECTED_TOP_KEYS = {
    "fault_location",
    "fault_symptoms",
    "fault_reason",
    "fault_measures",
    "resolution_status",
}

EXPECTED_LOCATION_KEYS = {"name", "machine"}


def analyze_schema(result):
    missing = 0
    extra = 0
    type_error = 0
    enum_violation = 0

    if not isinstance(result, dict):
        return EXPECTED_FIELDS_PER_CASE, 0, EXPECTED_FIELDS_PER_CASE, ENUM_CHECKS_PER_CASE

    for key in EXPECTED_TOP_KEYS:
        if key not in result:
            missing += 1

    for key in result:
        if key not in EXPECTED_TOP_KEYS:
            extra += 1

    location = result.get("fault_location")

    if not isinstance(location, dict):
        missing += 2
        type_error += 2
        enum_violation += 1
    else:
        if "name" not in location:
            missing += 1
        elif not isinstance(location["name"], str):
            type_error += 1

        if "machine" not in location:
            missing += 1
        else:
            machine = location["machine"]
            if machine is not None and not isinstance(machine, str):
                type_error += 1
            elif machine is not None and machine not in VALID_MACHINES:
                enum_violation += 1

        for key in location:
            if key not in EXPECTED_LOCATION_KEYS:
                extra += 1

    symptoms = result.get("fault_symptoms")

    if symptoms is None:
        missing += 1
    elif not isinstance(symptoms, list):
        type_error += 1
    else:
        for symptom in symptoms:
            if not isinstance(symptom, str):
                type_error += 1

    reasons = result.get("fault_reason")

    if reasons is None:
        missing += 1
    elif not isinstance(reasons, list):
        type_error += 1
    else:
        for reason in reasons:
            if not isinstance(reason, dict):
                type_error += 1
            elif "name" not in reason or not isinstance(reason["name"], str):
                type_error += 1

            if isinstance(reason, dict):
                for key in reason:
                    if key != "name":
                        extra += 1

    measures = result.get("fault_measures")

    if measures is None:
        missing += 1
    elif not isinstance(measures, list):
        type_error += 1
    else:
        for measure in measures:
            if not isinstance(measure, dict):
                type_error += 1
            elif "description" not in measure or not isinstance(measure["description"], str):
                type_error += 1

            if isinstance(measure, dict):
                for key in measure:
                    if key != "description":
                        extra += 1

    resolution = result.get("resolution_status")

    if resolution is None:
        missing += 1
    elif not isinstance(resolution, str):
        type_error += 1
    elif resolution not in VALID_RESOLUTION:
        enum_violation += 1

    return missing, extra, type_error, enum_violation

In [ ]:
data = json.loads(prompt_file.read_text(encoding="utf-8"))

total_reports = len(data)
valid_cases = 0
json_parse_failures = 0

total_missing = 0
total_extra = 0
total_type_error = 0
total_enum_violation = 0

invalid_examples = []

for item in data:
    result = item.get("result")

    if result is None:
        json_parse_failures += 1

    missing, extra, type_error, enum_violation = analyze_schema(result)

    total_missing += missing
    total_extra += extra
    total_type_error += type_error
    total_enum_violation += enum_violation

    if missing == 0 and extra == 0 and type_error == 0 and enum_violation == 0:
        valid_cases += 1
    else:
        invalid_examples.append({
            "case_id": item.get("case_id"),
            "missing": missing,
            "extra": extra,
            "type_error": type_error,
            "enum_violation": enum_violation,
            "result": result,
            "raw_response": item.get("raw_response"),
        })

print("Prompt-only GPT-4o schema conformance")
print("=" * 60)
print(f"Total reports: {total_reports}")
print(f"JSON parse failures: {json_parse_failures}")
print(f"Valid cases: {valid_cases} / {total_reports}")
print(f"Missing fields: {total_missing} / {total_reports * EXPECTED_FIELDS_PER_CASE}")
print(f"Extra fields: {total_extra} / {total_reports * EXPECTED_FIELDS_PER_CASE}")
print(f"Type errors: {total_type_error} / {total_reports * EXPECTED_FIELDS_PER_CASE}")
print(f"Enumeration violations: {total_enum_violation} / {total_reports * ENUM_CHECKS_PER_CASE}")

Evaluation Logic for Prompt Only GPT

In [ ]:
VALID_RESOLUTION = {"Resolved", "Unknown", "N/A"}

EXPECTED_TOP_KEYS = {
    "fault_location",
    "fault_symptoms",
    "fault_reason",
    "fault_measures",
    "resolution_status",
}

EXPECTED_FIELDS_PER_CASE = 5
ENUM_CHECKS_PER_CASE = 1


def has_at_least_one_string_value(obj):
    if not isinstance(obj, dict):
        return False

    for value in obj.values():
        if isinstance(value, str) and value.strip():
            return True

    return False


def analyze_schema_structural(result):
    missing = 0
    extra = 0
    type_error = 0
    enum_violation = 0

    if not isinstance(result, dict):
        return EXPECTED_FIELDS_PER_CASE, 0, EXPECTED_FIELDS_PER_CASE, ENUM_CHECKS_PER_CASE

    # Top-level ontology fields
    for key in EXPECTED_TOP_KEYS:
        if key not in result:
            missing += 1

    # Extra top-level fields only
    for key in result:
        if key not in EXPECTED_TOP_KEYS:
            extra += 1

    # fault_location should be an object.
    # We do not enforce exact nested key names.
    location = result.get("fault_location")

    if location is None:
        missing += 1
    elif not isinstance(location, dict):
        type_error += 1
    elif not has_at_least_one_string_value(location):
        missing += 1

    # fault_symptoms should be list of strings.
    symptoms = result.get("fault_symptoms")

    if symptoms is None:
        missing += 1
    elif not isinstance(symptoms, list):
        type_error += 1
    else:
        for symptom in symptoms:
            if not isinstance(symptom, str):
                type_error += 1

    # fault_reason should be list of objects.
    # Each object should contain at least one string value.
    reasons = result.get("fault_reason")

    if reasons is None:
        missing += 1
    elif not isinstance(reasons, list):
        type_error += 1
    else:
        for reason in reasons:
            if not isinstance(reason, dict):
                type_error += 1
            elif not has_at_least_one_string_value(reason):
                missing += 1

    # fault_measures should be list of objects.
    # Each object should contain at least one string value.
    measures = result.get("fault_measures")

    if measures is None:
        missing += 1
    elif not isinstance(measures, list):
        type_error += 1
    else:
        for measure in measures:
            if not isinstance(measure, dict):
                type_error += 1
            elif not has_at_least_one_string_value(measure):
                missing += 1

    # resolution_status should be one of the allowed enum values.
    resolution = result.get("resolution_status")

    if resolution is None:
        missing += 1
    elif not isinstance(resolution, str):
        type_error += 1
    elif resolution not in VALID_RESOLUTION:
        enum_violation += 1

    return missing, extra, type_error, enum_violation

Terakhir extracted logs and 1 image, dan semua evaluation for schema conformance berhasil 100%. mungkin habis ini harus mengubah prompt nya untuk tidak terlalu detail.

In [ ]:
loose_cases = load_jsonl(DATA_DIR / "20_cases_for_baml_fewshot.jsonl")

loose_output_path = DATA_DIR / "prompt_only_gpt4o_20_cases_loose_one_by_one_raw_audit.json"

if loose_output_path.exists():
    loose_results = json.loads(loose_output_path.read_text(encoding="utf-8"))
    print("Existing extracted cases:", len(loose_results))
else:
    loose_results = []
    loose_output_path.write_text(
        json.dumps(loose_results, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    print("Created new file:", loose_output_path)

print("Total cases to extract:", len(loose_cases))

In [ ]:
def extract_one_prompt_only_case(case_no):
    """
    Extract one maintenance-log case using the current prompt-only function.
    case_no starts from 1.
    """

    if case_no < 1 or case_no > len(loose_cases):
        raise ValueError(f"case_no must be between 1 and {len(loose_cases)}")

    case = loose_cases[case_no - 1]
    log_text = case_to_log_text(case)
    case_id = case_id_from_log(log_text, f"case_{case_no}")

    prompt_only_results = json.loads(loose_output_path.read_text(encoding="utf-8"))

    existing_ids = {item["case_id"] for item in prompt_only_results}

    if case_id in existing_ids:
        print("Already extracted:", case_id)
        return case_id

    print(f"Extracting case {case_no}/{len(loose_cases)}: {case_id}")

    raw_output = call_gpt4o([
        {
            "role": "system",
            "content": "You extract fault information from maintenance logs."
        },
        {
            "role": "user",
            "content": build_log_prompt(log_text)
        }
    ])

    try:
        json.loads(raw_output)
        raw_json_valid = True
    except Exception:
        raw_json_valid = False

    cleaned_parsed = parse_json_or_none(raw_output)
    cleaned_json_valid = cleaned_parsed is not None

    record = {
        "case_no": case_no,
        "case_id": case_id,
        "raw_output": raw_output,
        "raw_json_valid": raw_json_valid,
        "cleaned_json_valid": cleaned_json_valid,
        "result": cleaned_parsed
    }

    prompt_only_results.append(record)

    loose_output_path.write_text(
        json.dumps(prompt_only_results, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    print("Saved:", case_id)
    print("Raw JSON valid:", raw_json_valid)
    print("Cleaned JSON valid:", cleaned_json_valid)

    return case_id

In [ ]:
def evaluate_one_prompt_only_case(case_id):
    prompt_only_results = json.loads(loose_output_path.read_text(encoding="utf-8"))

    matches = [
        item for item in prompt_only_results
        if item["case_id"] == case_id
    ]

    if not matches:
        raise ValueError(f"Case not found: {case_id}")

    item = matches[0]
    
    if item["raw_json_valid"]:
        result = json.loads(item["raw_output"])
    else:
        result = None

    missing, extra, type_error, enum_violation = analyze_schema_structural(result)

    valid_case = (
        missing == 0
        and extra == 0
        and type_error == 0
        and enum_violation == 0
    )

    print("=" * 100)
    print("Case:", item["case_id"])
    print("=" * 100)
    print("Raw JSON valid:", item["raw_json_valid"])
    print("Cleaned JSON valid:", item["cleaned_json_valid"])
    print("Valid case:", valid_case)
    print()
    print("Missing fields:", missing)
    print("Extra fields:", extra)
    print("Type errors:", type_error)
    print("Enum violations:", enum_violation)
    print()
    print("Raw output:")
    print(item["raw_output"])
    print()
    print("Parsed result:")
    print(json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
case_id = extract_one_prompt_only_case(1)
evaluate_one_prompt_only_case(case_id)

In [ ]:
case_id = extract_one_prompt_only_case(2)
evaluate_one_prompt_only_case(case_id)

In [ ]:
case_id = extract_one_prompt_only_case(3)
evaluate_one_prompt_only_case(case_id)

In [ ]:
case_id = extract_one_prompt_only_case(4)
evaluate_one_prompt_only_case(case_id)

In [ ]:
case_id = extract_one_prompt_only_case(5)
evaluate_one_prompt_only_case(case_id)

In [ ]:
case_id = extract_one_prompt_only_case(6)
evaluate_one_prompt_only_case(case_id)

In [ ]:
case_id = extract_one_prompt_only_case(7)
evaluate_one_prompt_only_case(case_id)

In [ ]:
case_id = extract_one_prompt_only_case(8)
evaluate_one_prompt_only_case(case_id)

In [ ]:
case_id = extract_one_prompt_only_case(9)
evaluate_one_prompt_only_case(case_id)

In [ ]:
case_id = extract_one_prompt_only_case(10)
evaluate_one_prompt_only_case(case_id)

In [ ]:
case_id = extract_one_prompt_only_case(11)
evaluate_one_prompt_only_case(case_id)

In [ ]:
case_id = extract_one_prompt_only_case(12)
evaluate_one_prompt_only_case(case_id)

In [ ]:
case_id = extract_one_prompt_only_case(13)
evaluate_one_prompt_only_case(case_id)

In [ ]:
case_id = extract_one_prompt_only_case(14)
evaluate_one_prompt_only_case(case_id)

In [ ]:
case_id = extract_one_prompt_only_case(15)
evaluate_one_prompt_only_case(case_id)

In [ ]:
case_id = extract_one_prompt_only_case(16)
evaluate_one_prompt_only_case(case_id)

In [ ]:
case_id = extract_one_prompt_only_case(17)
evaluate_one_prompt_only_case(case_id)

In [ ]:
case_id = extract_one_prompt_only_case(18)
evaluate_one_prompt_only_case(case_id)

In [ ]:
case_id = extract_one_prompt_only_case(19)
evaluate_one_prompt_only_case(case_id)

In [ ]:
case_id = extract_one_prompt_only_case(20)
evaluate_one_prompt_only_case(case_id)

PROMPT ONLY EXTRACTION (BATCH)

In [ ]:
def batch_extract_prompt_only_logs(input_filename, output_filename):
    cases = load_jsonl(DATA_DIR / input_filename)
    output_path = DATA_DIR / output_filename

    results = []

    for idx, case in enumerate(cases, start=1):
        log_text = case_to_log_text(case)
        case_id = case_id_from_log(log_text, f"case_{idx}")

        print(f"Processing {idx}/{len(cases)}: {case_id}")

        try:
            raw_output = call_gpt4o([
                {
                    "role": "system",
                    "content": "You extract fault information from maintenance logs."
                },
                {
                    "role": "user",
                    "content": build_log_prompt(log_text)
                }
            ])

            try:
                json.loads(raw_output)
                raw_json_valid = True
            except Exception:
                raw_json_valid = False

            cleaned_parsed = parse_json_or_none(raw_output)

            results.append({
                "case_no": idx,
                "case_id": case_id,
                "raw_output": raw_output,
                "raw_json_valid": raw_json_valid,
                "cleaned_json_valid": cleaned_parsed is not None,
                "result": cleaned_parsed
            })

        except Exception as exc:
            results.append({
                "case_no": idx,
                "case_id": case_id,
                "raw_output": None,
                "raw_json_valid": False,
                "cleaned_json_valid": False,
                "result": None,
                "error": str(exc)
            })

        output_path.write_text(
            json.dumps(results, indent=2, ensure_ascii=False),
            encoding="utf-8"
        )

    print("Saved:", output_path)
    return output_path

In [ ]:
batch_20_path = batch_extract_prompt_only_logs(
    input_filename="20_cases_for_baml_fewshot.jsonl",
    output_filename="prompt_only_gpt4o_20_cases_batch_raw_audit.json"
)

In [ ]:
batch_remaining_path = batch_extract_prompt_only_logs(
    input_filename="remaining_cases_for_baml_fewshot.jsonl",
    output_filename="prompt_only_gpt4o_remaining_cases_batch_raw_audit.json"
)

In [ ]:
import pandas as pd

def evaluate_batch_raw_output(path):
    data = json.loads(Path(path).read_text(encoding="utf-8"))

    rows = []

    for item in data:
        if item["raw_json_valid"]:
            result = json.loads(item["raw_output"])
        else:
            result = None

        missing, extra, type_error, enum_violation = analyze_schema_structural(result)

        valid_case = (
            missing == 0
            and extra == 0
            and type_error == 0
            and enum_violation == 0
        )

        rows.append({
            "case_no": item["case_no"],
            "case_id": item["case_id"],
            "raw_json_valid": item["raw_json_valid"],
            "cleaned_json_valid": item["cleaned_json_valid"],
            "valid_case": valid_case,
            "missing_fields": missing,
            "extra_fields": extra,
            "type_errors": type_error,
            "enum_violations": enum_violation,
        })

    df = pd.DataFrame(rows)

    display(df)

    print("Reports:", len(df))
    print("Raw JSON valid:", df["raw_json_valid"].sum(), "/", len(df))
    print("Cleaned JSON valid:", df["cleaned_json_valid"].sum(), "/", len(df))
    print("Valid cases:", df["valid_case"].sum(), "/", len(df))
    print("Missing fields:", df["missing_fields"].sum())
    print("Extra fields:", df["extra_fields"].sum())
    print("Type errors:", df["type_errors"].sum())
    print("Enum violations:", df["enum_violations"].sum())

    return df

In [ ]:
batch_20_eval_df = evaluate_batch_raw_output(batch_20_path)